In [1]:
import numpy as np
from typing import Dict, List, Tuple, Optional, Any

class ChessRelaxationLabeler:
    """
    Subclass for chess role-assignment and tactical constraint reasoning.
    Implements standard HRL (Pieces -> Roles) alongside Dual Inversion (Roles -> Pieces),
    algebraic piece constraints, and multi-scale board state-space pruning.
    """
    def __init__(self, board_size: int = 8, active_pieces_count: int = 32):
        self.N = board_size  # Default 8 for standard, scales down to 6, 4, 2
        self.num_objects = active_pieces_count

        # 1. Primary Framework: Objects = Pieces (ID/Type), Labels = Tactical Roles
        self.primary_labels = [
            "King-Shield", "Infiltrator", "Blockader", "Open-File-Rook",
            "Diagonal-Lurker", "Escape-Route-Sealer", "Null-Noise"
        ]

        # 2. Dual Framework: Objects = Tactical Roles Needed, Labels = Available Pieces
        self.dual_labels = [
            "White-King", "White-Queen", "White-Rook", "White-Bishop", "White-Knight", "White-Pawn",
            "Black-King", "Black-Queen", "Black-Rook", "Black-Bishop", "Black-Knight", "Black-Pawn"
        ]

        # Initialize Probability Tensors for the dual-relaxation pipelines
        self.P_primary = np.ones((self.num_objects, len(self.primary_labels))) / len(self.primary_labels)
        self.P_dual = np.ones((len(self.primary_labels), len(self.dual_labels))) / len(self.dual_labels)

        # Static database of solved subsets (M x M boards) to anchor edge relaxation
        self.solved_database: Dict[str, Any] = {}

    def register_solved_subproblem(self, board_signature: str, evaluation_value: float):
        """Registers a verified solution for an M x M board fragment to clip open plies."""
        self.solved_database[board_signature] = evaluation_value

    def verify_algebraic_constraints(self, piece_type: str, legal_matrix: np.ndarray) -> bool:
        """
        Enforces piece definitions via algebraic constraints (e.g. Queen = Rook + Bishop).
        Verifies that composite fields do not violate underlying capabilities.
        """
        # Extract directional masks
        lateral_mask = self._get_movement_template("Rook")
        diagonal_mask = self._get_movement_template("Bishop")
        queen_mask = self._get_movement_template("Queen")

        if piece_type == "Queen":
            # Algebraic verification: Queen capabilities must equal the union of Rook and Bishop
            return np.array_equal(queen_mask, lateral_mask | diagonal_mask)
        elif piece_type == "King":
            # King acts as a bounded radius=1 Queen
            return np.all(legal_matrix <= queen_mask) and np.max(np.sum(legal_matrix, axis=0)) <= 1
        return True

    def calculate_hunting_support(self, board_state: np.ndarray, target_sq: Tuple[int, int], hunting_lens: str = "checkmate") -> np.ndarray:
        """
        Calculates local support arrays treating target captures through a checkmate/stalemate lens.
        Determines the containment bounds (escape squares completely blocked or covered).
        """
        x, y = target_sq
        escape_squares = self._get_immediate_neighborhood(x, y)
        support_matrix = np.zeros((self.N, self.N))

        # For a piece to be locked down (hunted), its escape squares must converge to zero utility
        for sq in escape_squares:
            if hunting_lens == "checkmate":
                # Support increases as escape avenues are systemically choked off by allies
                support_matrix[sq] += self._compute_inhibitory_coverage(board_state, sq)
            elif hunting_lens == "stalemate":
                # Stalemating constraints ensure no active legal alternative moves remain
                support_matrix[sq] += self._compute_immobility_fields(board_state, sq)
        return support_matrix

    def execute_dual_relaxation_step(self, compatibility_tensor: np.ndarray, adjacency_weights: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Executes a single step of parallel relaxation.
        Updates the primary distribution (Pieces->Roles) and the dual distribution (Roles->Pieces).
        """
        # 1. Update Primary Relaxation Loop
        Q_primary = np.zeros_like(self.P_primary)
        for i in range(self.num_objects):
            for lam_idx, lam in enumerate(self.primary_labels):
                # Aggregate spatial neighbors and tactical role compatibilities
                support = 0.0
                for j in range(self.num_objects):
                    if i != j:
                        edge_w = adjacency_weights[i, j]
                        inner_sum = np.sum(compatibility_tensor[i, j, lam_idx, :] * self.P_primary[j, :])
                        support += edge_w * inner_sum
                Q_primary[i, lam_idx] = support

        # Standard multiplicative update mapping to the probability simplex
        self.P_primary = self.P_primary * (1.0 + Q_primary)
        self.P_primary /= np.sum(self.P_primary, axis=1, keepdims=True)

        # 2. Update Dual Inversion Loop (Roles are now independent targets, Pieces are labels)
        # Interlocks current assignments with the primary matrix state
        self.P_dual = self.P_dual * (1.0 + np.dot(self.P_primary.T, self.P_primary))
        self.P_dual /= np.sum(self.P_dual, axis=1, keepdims=True)

        return self.P_primary, self.P_dual

    def _get_movement_template(self, piece: str) -> np.ndarray:
        # Internal template generation for algebraic validations
        arr = np.zeros((3, 3), dtype=bool)
        if piece == "Rook" or piece == "Queen":
            arr[1, :] = True; arr[:, 1] = True
        if piece == "Bishop" or piece == "Queen":
            np.fill_diagonal(arr, True); np.fliplr(arr).flat[::4] = True
        arr[1, 1] = False
        return arr

    def _get_immediate_neighborhood(self, x: int, y: int) -> List[Tuple[int, int]]:
        res = []
        for i in [-1, 0, 1]:
            for j in [-1, 0, 1]:
                if 0 <= x+i < self.N and 0 <= y+j < self.N and not (i == 0 and j == 0):
                    res.append((x+i, y+j))
        return res

    def _compute_inhibitory_coverage(self, b: np.ndarray, sq: Tuple[int, int]) -> float: return 1.47
    def _compute_immobility_fields(self, b: np.ndarray, sq: Tuple[int, int]) -> float: return 0.88
